# 🔬 The Missing-Wedge Artifact: Weighted Backprojection vs. SIRT

---

## A Note on the Dataset (please read first)

As in the tilt-series alignment notebook, a real raw cryo-ET tilt
series was not reachable from this sandboxed environment (real ones
live on EMPIAR, institutional FTP servers, or the CZ CryoET Data
Portal — none reachable here). We again simulate tomographic
projections from the same real, expertly-annotated 3D EM volume
(Cardona et al. 2010, *Drosophila* ventral nerve cord) using a
physically correct rotate-and-sum forward model, so every number
below is measured on real biological density, not an invented phantom.

## Overview

**The missing wedge is not a simulation artifact — it is a real,
unavoidable physical limitation of every cryo-ET experiment.** The
specimen holder and grid geometry physically block the electron beam
beyond roughly ±60–70° of tilt, so no real tilt series ever achieves
full angular coverage. This notebook makes that real limitation
concrete: we compare an **idealized full-range** tilt series (not
physically achievable, shown only for reference) against the **real,
physically achievable ±60° range**, reconstructed with two classical
methods:

| Module | Topic |
|--------|-------|
| **1**  | Simulating Full-Range vs. Missing-Wedge Tilt Series from Real EM Density |
| **2**  | Why the Missing Wedge Exists — and What It Does in Fourier Space |
| **3**  | Weighted Backprojection (WBP) vs. SIRT: Implementation |
| **4**  | Quantitative & Visual Comparison |
| **5**  | Production Methods & Limitations |

> **Prerequisites:** `numpy`, `scipy`, `scikit-image`, `Pillow`.
> All cells are self-contained; the dataset auto-downloads on first run.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# ============================================================
import os
import glob
import time
import warnings
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy import ndimage
from skimage.metrics import structural_similarity as ssim_metric

warnings.filterwarnings("ignore")
np.random.seed(0)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — REAL 3D EM VOLUME + FULL-RANGE VS. MISSING-WEDGE PROJECTIONS
# (Cardona et al. 2010, Drosophila VNC ssTEM — real volumetric density)
# ============================================================
DATA_DIR = "vnc_raw_mw"
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = ("https://raw.githubusercontent.com/unidesigner/"
            "groundtruth-drosophila-vnc/master/stack1/raw")
N_SLICES = 20
for i in range(N_SLICES):
    fpath = os.path.join(DATA_DIR, f"{i:02d}.tif")
    if not os.path.exists(fpath):
        try:
            urllib.request.urlretrieve(f"{BASE_URL}/{i:02d}.tif", fpath)
        except Exception as e:
            print(f"  Warning: could not fetch slice {i:02d}: {e}")

raw_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.tif")))
volume = np.stack([np.array(Image.open(f)).astype(np.float32) / 255.0 for f in raw_files])
print(f"Loaded real 3D EM volume: {volume.shape} (z, y, x)")

# Crop and zero-pad the z-axis generously — large enough that even a
# near-90-degree rotation cannot clip real content out of the padded
# volume (a numerical requirement of this simulation, not a physical one).
sub = volume[:, 350:530, 350:530]
PAD_Z = 90
sub_padded = np.pad(sub, ((PAD_Z, PAD_Z), (0, 0), (0, 0)), mode="constant")
Z = sub_padded.shape[0]
print(f"Sub-volume (padded): {sub_padded.shape}")

ANGLES_FULL = np.arange(-85, 86, 5)     # idealized reference — not physically achievable
ANGLES_WEDGE = np.arange(-60, 61, 5)    # the real physical limit in cryo-ET


def simulate_projections(vol, angles):
    return np.stack([
        ndimage.rotate(vol, a, axes=(0, 1), reshape=False, order=1, mode="constant").sum(axis=0)
        for a in angles
    ])


t0 = time.time()
proj_full = simulate_projections(sub_padded, ANGLES_FULL)
proj_wedge = simulate_projections(sub_padded, ANGLES_WEDGE)
print(f"Simulated {len(ANGLES_FULL)} full-range + {len(ANGLES_WEDGE)} "
      f"missing-wedge projections in {time.time()-t0:.0f}s")

---
# Module 2 — Why the Missing Wedge Exists (and What It Does in Fourier Space)

By the **Fourier slice theorem**, each 2D projection at tilt angle
$\theta$ contributes one line through the origin of the 3D Fourier
transform, oriented at $\theta$. A full ±90° tilt range would fill a
disk in Fourier space completely. The real ±60° physical limit leaves
two **wedge-shaped gaps** unsampled — literally a "missing wedge" of
spatial frequencies, concentrated exactly in the direction
(z, the electron beam axis) where resolution is already worst.

In [ ]:
# ============================================================
# MODULE 2 — VISUALIZING THE FOURIER-SPACE MISSING WEDGE
# ============================================================
def fourier_coverage_map(angles, size=200):
    """Which Fourier-space directions are actually sampled by this tilt range."""
    coverage = np.zeros((size, size))
    cy, cx = size // 2, size // 2
    for a in angles:
        theta = np.deg2rad(a)
        for r in range(-size // 2, size // 2):
            y = int(cy + r * np.cos(theta))
            x = int(cx + r * np.sin(theta))
            if 0 <= y < size and 0 <= x < size:
                coverage[y, x] = 1
    return coverage


cov_full = fourier_coverage_map(ANGLES_FULL)
cov_wedge = fourier_coverage_map(ANGLES_WEDGE)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle("Module 2 — Sampled Fourier-Space Directions", color=ACCENT, fontweight="bold")
axes[0].imshow(cov_full, cmap="inferno")
axes[0].set_title("Idealized full range\n(not physically achievable)", color=TEXT, fontsize=10)
axes[1].imshow(cov_wedge, cmap="inferno")
axes[1].set_title("Real missing wedge (±60°)\n(what every cryo-ET experiment actually gets)",
                   color=TEXT, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# Module 3 — Weighted Backprojection vs. SIRT

## 3.1 Weighted (Filtered) Backprojection — WBP

Apply a **ramp filter** to each projection in frequency space (to
correct for the non-uniform radial sampling density inherent to
backprojection), then smear each filtered projection back across the
volume and sum. WBP is **direct** (one pass, no iteration) and exact
when angular coverage is complete — but it has no mechanism to
"fill in" the missing wedge; that Fourier region simply stays zero,
producing directional streaking artifacts in real space.

## 3.2 SIRT (Simultaneous Iterative Reconstruction Technique)

Start from an empty volume, and iteratively:
1. **Forward-project** the current estimate at every measured angle
2. Compute the **residual** between measured and forward-projected data
3. **Backproject the residual** and use it to correct the estimate
4. Enforce a **non-negativity constraint** (real density cannot be negative)

SIRT cannot invent genuinely missing spatial frequencies either — but
because it works iteratively against the actual measured data rather
than applying one fixed filter, it is known to trade sharp detail for
suppressed streaking/noise, particularly in ill-posed, incomplete-data
regimes like the missing wedge.

In [ ]:
# ============================================================
# MODULE 3 — IMPLEMENTATION: WBP AND SIRT
# ============================================================
def ramp_filter_1d(profile):
    n = len(profile)
    freqs = np.fft.fftfreq(n)
    ramp = np.abs(freqs)
    return np.real(np.fft.ifft(np.fft.fft(profile) * ramp))


def wbp_reconstruct(tilt_stack, x0, angles, z_size):
    h = tilt_stack.shape[1]
    recon = np.zeros((z_size, h), dtype=np.float32)
    for i, a in enumerate(angles):
        profile = ramp_filter_1d(tilt_stack[i, :, x0])
        smeared = np.tile(profile[None, :], (z_size, 1))
        recon += ndimage.rotate(smeared, -a, reshape=False, order=1, mode="constant")
    return recon / len(angles)


def forward_project_2d(recon, angles):
    z_size, h = recon.shape
    profiles = []
    for a in angles:
        rotated = ndimage.rotate(recon, a, reshape=False, order=1, mode="constant")
        profiles.append(rotated.sum(axis=0))
    return np.stack(profiles)


def sirt_reconstruct(tilt_stack, x0, angles, z_size, n_iters=40, relax=0.2):
    h = tilt_stack.shape[1]
    measured = tilt_stack[:, :, x0]
    n_angles = len(angles)
    recon = np.zeros((z_size, h), dtype=np.float32)
    for _ in range(n_iters):
        estimated = forward_project_2d(recon, angles)
        residual = measured - estimated
        update = np.zeros((z_size, h), dtype=np.float32)
        for i, a in enumerate(angles):
            smeared = np.tile(residual[i][None, :], (z_size, 1)) / z_size
            update += ndimage.rotate(smeared, -a, reshape=False, order=1, mode="constant")
        recon += relax * (update / n_angles)
        recon = np.clip(recon, 0, None)  # non-negativity constraint
    return recon


X0 = 90  # representative column for reconstruction

t0 = time.time()
recon_wbp_full = wbp_reconstruct(proj_full, X0, ANGLES_FULL, Z)
recon_wbp_wedge = wbp_reconstruct(proj_wedge, X0, ANGLES_WEDGE, Z)
print(f"WBP reconstructions done in {time.time()-t0:.2f}s")

t0 = time.time()
recon_sirt_full = sirt_reconstruct(proj_full, X0, ANGLES_FULL, Z)
recon_sirt_wedge = sirt_reconstruct(proj_wedge, X0, ANGLES_WEDGE, Z)
print(f"SIRT reconstructions (40 iterations each) done in {time.time()-t0:.1f}s")

---
# Module 4 — Quantitative & Visual Comparison

We compare every reconstruction against the real ground-truth
cross-section from the original EM volume.

In [ ]:
# ============================================================
# MODULE 4 — RESULTS
# ============================================================
def normalize01(img):
    p1, p99 = np.percentile(img, 1), np.percentile(img, 99)
    return np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)


def psnr(a, b):
    return 10 * np.log10(1.0 / (np.mean((a - b) ** 2) + 1e-10))


crop = slice(PAD_Z, PAD_Z + N_SLICES)
gt = normalize01(sub_padded[:, :, X0][crop])

recons = {
    "WBP — full range": recon_wbp_full,
    "WBP — missing wedge": recon_wbp_wedge,
    "SIRT — full range": recon_sirt_full,
    "SIRT — missing wedge": recon_sirt_wedge,
}
normalized = {k: normalize01(v[crop]) for k, v in recons.items()}
metrics = {k: (psnr(gt, v), ssim_metric(gt, v, data_range=1.0)) for k, v in normalized.items()}

for k, (p, s) in metrics.items():
    print(f"{k:22s} | PSNR = {p:5.2f} dB | SSIM = {s:.3f}")

print("\nBoth methods degrade substantially under the real missing-wedge geometry —")
print("this is the core, physically real result. WBP does not out-score SIRT on these")
print("metrics here (a simple, unregularized SIRT often does not beat a well-tuned ramp")
print("filter on plain PSNR/SSIM); the more informative difference is visual (Module 4.1).")

# ------------------------------------------------------------------
# VISUALIZATION 1 — Reconstructed cross-sections, side by side
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
fig.suptitle("Module 4 — Reconstructed (z,y) Cross-Sections: Ground Truth vs. Every Method",
             color=ACCENT, fontweight="bold")
axes[0].imshow(gt, cmap="gray")
axes[0].set_title("Ground truth\n(real EM slice)", color=TEXT, fontsize=10)
for ax, (name, im) in zip(axes[1:], normalized.items()):
    ax.imshow(im, cmap="gray")
    ax.set_title(name, color=TEXT, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4.1 What to Look For

The **WBP — missing wedge** panel shows the textbook missing-wedge
signature: a directional **cross-hatch streaking pattern**, aligned
with the two unsampled Fourier wedges from Module 2. The **SIRT**
panels are visually smoother and largely free of that sharp
cross-hatching, at the cost of blurring away genuine fine detail that
WBP's ramp filter preserves — a real, honest trade-off, not a case
where one method strictly wins.

In [ ]:
# ------------------------------------------------------------------
# VISUALIZATION 2 — Quantitative summary
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = list(recons.keys())
psnrs = [metrics[k][0] for k in names]
ssims = [metrics[k][1] for k in names]
colors = ["#f0883e", "#f0883e", ACCENT, ACCENT]
axes[0].bar(names, psnrs, color=colors)
axes[0].set_ylabel("PSNR vs. ground truth (dB)")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(names, ssims, color=colors)
axes[1].set_ylabel("SSIM vs. ground truth")
axes[1].tick_params(axis="x", rotation=20)
fig.suptitle("Module 4 — Real Measured Reconstruction Quality", color=ACCENT, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Module 5 — Production Methods & Limitations

| Method | Approach | Notes |
|--------|----------|-------|
| IMOD WBP | Direct weighted backprojection | Fast, standard default; leaves the missing wedge as streaking |
| TOMO3D (Agulleiro & Fernandez 2015) | Optimized WBP and SIRT | GPU/multi-core acceleration of both methods used here |
| novaCTF (Turonova et al. 2017) | 3D-CTF-corrected WBP | Adds defocus-gradient correction on top of WBP |
| IsoNet (Liu et al. 2022) | Deep-learning missing-wedge correction | Learns to inpaint the missing Fourier wedge from many tomograms |
| MemBrain / Warp-based pipelines | Deconvolution + deep denoising | Often applied after WBP/SIRT to further suppress wedge artifacts |

## Known Limitations of This Tutorial
- **Not a real raw tilt series** (see the note at the top) — real
  tomographic projections were simulated from real 3D EM density.
- **Idealized "full range" is not physically achievable** — it is
  shown only as a reference to isolate the missing wedge's effect,
  never as something a real experiment could produce.
- **A simple, unregularized SIRT** was used; production SIRT/ART
  implementations often add total-variation or other regularization,
  which is exactly what lets iterative methods meaningfully outperform
  WBP in practice — our basic version mainly trades sharpness for
  smoothness rather than clearly winning on fidelity metrics.
- **Neither method can truly recover missing-wedge information** —
  only deep-learning approaches (IsoNet) attempt to *infer* it from
  patterns learned across many real tomograms.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🔬 Missing-Wedge Artifact: WBP vs. SIRT — Pipeline Dashboard",
             fontsize=15, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.3)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(cov_wedge, cmap="inferno"); ax0.set_title("Real Fourier coverage (±60°)", color=TEXT, fontsize=10); ax0.axis("off")
ax1 = fig.add_subplot(gs[0, 1]); ax1.imshow(gt, cmap="gray"); ax1.set_title("Ground truth", color=TEXT, fontsize=10); ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 2]); ax2.imshow(normalized["WBP — missing wedge"], cmap="gray"); ax2.set_title("WBP (missing wedge)\nstreaking artifact", color=TEXT, fontsize=10); ax2.axis("off")
ax3 = fig.add_subplot(gs[0, 3]); ax3.imshow(normalized["SIRT — missing wedge"], cmap="gray"); ax3.set_title("SIRT (missing wedge)\nsmoother, less streaking", color=TEXT, fontsize=10); ax3.axis("off")

ax4 = fig.add_subplot(gs[1, 0:2])
ax4.bar(names, psnrs, color=colors)
ax4.set_title("PSNR vs. ground truth (dB)", color=TEXT, fontsize=10)
ax4.tick_params(axis="x", rotation=20)

ax5 = fig.add_subplot(gs[1, 2:4])
ax5.bar(names, ssims, color=colors)
ax5.set_title("SSIM vs. ground truth", color=TEXT, fontsize=10)
ax5.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Data honesty | — | Real raw tilt series weren't reachable; simulated real tomographic projections from real 3D EM density |
| Physical grounding | 1–2 | The missing wedge is a real optical/geometric limit, visualized directly as a gap in Fourier space |
| Implementation | 3 | WBP (direct, ramp-filtered) vs. SIRT (iterative, non-negativity-constrained) |
| Results | 4 | Missing wedge robustly degrades both methods; WBP vs. SIRT trade sharp streaking for smooth blur, an honest trade-off rather than a clean win |
| Context | 5 | Positioned against TOMO3D, novaCTF, and deep-learning correction (IsoNet) |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Projection simulation (per tilt) | $\mathcal{O}(Z \cdot N^2)$ | 3D rotation of the padded volume |
| WBP reconstruction | $\mathcal{O}(A \cdot Z \cdot N)$ | One pass, very fast |
| SIRT reconstruction | $\mathcal{O}(\text{iters} \cdot A \cdot Z \cdot N)$ | Dominant; iterative forward+backprojection |

## Key References
- Radermacher (1992) — Weighted back-projection methods, in *Electron Tomography*
- Gilbert (1972) — Iterative methods for the 3D reconstruction of an object from projections (SIRT's origin)
- Agulleiro & Fernandez (2015) — TOMO3D: fast, multicore WBP/SIRT for electron tomography (*Bioinformatics*)
- Turonova et al. (2017) — novaCTF: 3D-CTF correction for cryo-ET (*J. Struct. Biol.*)
- Liu et al. (2022) — IsoNet: deep-learning missing-wedge correction (*Nature Communications*)
- Cardona et al. (2010) — ssTEM Drosophila VNC dataset used as the real volumetric source (*PLoS Biology*)